In [9]:
!pip install torch --index-url https://download.pytorch.org/whl/cu121


Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp313-cp313-linux_x86_64.whl (780.4 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 187.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 101.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 183.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 43.2 MB/s  0:00:07:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 51.0 MB/s  0:00:05:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 40.8 MB/s  0:00:02:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 163.1 MB/s  0:00:00eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 139.6 MB/s  0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 118.4 MB/s  0:00:0100:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# environment

import torch
import json
import chromadb
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_PATH = "/space_mounts/pars/RAG_Project"
DB_PATH = f"{BASE_PATH}/chroma_db"
METADATA_PATH = f"{BASE_PATH}/target_metadata.json"
SAVE_PATH = f"{BASE_PATH}/probing_results.json"

# load model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [3]:
# metadata
DB_PATH = "./chroma_db"
chroma_client = chromadb.PersistentClient(path=DB_PATH)

heartbeat = chroma_client.heartbeat()
print(f"✅ {heartbeat}")

collections = {
    "clean": chroma_client.get_collection(name="clean_set"),
    "naive": chroma_client.get_collection(name="naive_poison_set"),
    "targeted": chroma_client.get_collection(name="targeted_poison_set")
}
with open(METADATA_PATH, "r") as f:
    target_questions = json.load(f)

print(f"✅ ")

✅ 1778429053893637098
✅ 


In [4]:
import torch.nn.functional as F

def run_logit_lens(question, context, model, tokenizer):
    prompt = f"Context: {context}\nQuestion: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        all_hidden_states = outputs.hidden_states

    lm_head = model.get_output_embeddings()
    final_norm = model.model.norm
    
    layer_decoded = []
    for h_state in all_hidden_states:
        h_norm = final_norm(h_state)
        logits = lm_head(h_norm)
        last_token_logits = logits[0, -1, :]
        probs = F.softmax(last_token_logits, dim=-1)
        word = tokenizer.decode([torch.argmax(probs).item()]).strip()
        layer_decoded.append(word)
    return layer_decoded

In [6]:
from tqdm import tqdm

results = []

for i, item in enumerate(tqdm(target_questions)):
    q_id = item['id']
    q_text = item['question']
    
    try:
        # Context
        c_doc = collections['clean'].get(ids=[f"doc_{i}"])['documents'][0]
        n_doc = collections['naive'].get(ids=[f"naive_poison_{q_id}"])['documents'][0]
        t_doc = collections['targeted'].get(ids=[f"targeted_poison_{q_id}"])['documents'][0]
        
        # comparison
        results.append({
            "id": q_id,
            "question": q_text,
            "ground_truth": item['answer'],
            "minds": {
                "clean": run_logit_lens(q_text, c_doc, model, tokenizer),
                "naive": run_logit_lens(q_text, n_doc, model, tokenizer),
                "targeted": run_logit_lens(q_text, t_doc, model, tokenizer)
            }
        })
    except Exception as e:
        print(f"⚠️ skip {q_id}: {str(e)}")

with open(SAVE_PATH, "w", encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"✅")

100%|██████████| 50/50 [07:59<00:00,  9.58s/it]

✅
